<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week01_1_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 수상작 리뷰

# 수상작 소개

**의류 제조 회사 생산성 예측 AI 해커톤**  

(https://dacon.io/competitions/official/235986/codeshare/6996?page=1&dtype=recent)

목표: 의류 제조 회사의 생산성 데이터를 이용해 예측 모델을 개발

설명; 의류 제조 공정 데이터로 생산성을 회귀 예측



---


## 코드 흐름 요약

- 모델 성능:  CAT > GBR > NGB > XGB > ETR > RF
- 앙상블  
  - Kfold 사용시 성능이 떨어져 따로 튜닝하지 않음
  - voting시 soft voting기법 사용

### 코드 흐름

1. 라이브러리 임포트
- 데이터 핸들링 툴
- 데이터 시각화 툴
- seed 설정

2. 데이터 로드
- 칼럼 info 확인
- duplicate 확인
- NA 확인

*EDA 시 이상치 있으나 데이터 샘플 수 적어 이상치 제거 안 함

3. 데이터 전처리
- NAN은 평균으로 대체(Train의 평균으로 대체해 Data Leakage 방지)
- Encoding(마찬가지, Train 데이터로 Fitting된 Label Encoder에서 transform만 수행)
- 로그변환
  - 타겟피처 로그화, 정규분포에 좀 더 가깝도록 변형
  - Skew 확인, 0.75 이상인 피처
- 피처변환: 스케일링, 피처명에 Scaled 붙여 변경

4. 모델링
- X,y 변수 지정. dataset copy해 사용
- 평가지수 'NMAE', NMAE 함수 생성 및 make_scorer로 낮은 값이 더 좋은 평가지수 라는 로직 입력
- RF, 결정트리, GBR, ETR, XGB, LGBM, 엘라스틱넷, AdaBoostRegressor, CAT, Pool, NGB, HistGBR, LR, Lasso, Ridge 모두 학습 및 예측 진행.
  - fit, predict, NMAE 함수 입력
- 좋은 결과를 가지는 모델만 8개 선정
- kfold로 학습 시의 성능 검증 : 성능 훨씬 감소 -> kfold 전 모델 성능 사용 채택
- LGB, HGB 제외하고 소프트 보팅하여 최종 결과물 도출


## 새롭게 알게 된 내용 / 어려운 내용 / 배울 점 등 정리 및 주요 코드 3줄 이상 작성

- 전처리 시 데이터 누수 방지를 위해 Train data 기준으로 변형해주는 것 주의
- skew 확인 후 원래 범주형 변수였으며 수치 간 범위가 적은 피처를 제외하고 확인 및 log화 진행
- 모델링 전 X,y 별도 지정... 기존 데이터셋 copy해 사용하면 코드 전체 다시 수행할 일이 없을 것. 데이터 규모 작을때 유용한 방식


In [1]:
# X = train_x.copy()
# y = np.log1p(train_y)
# X_test0 = test.copy()

- 평가지수가 'NMAE'인데, NMAE 함수는 sklearn에서 기본 제공하는 표준평가지표가 아님. 별도로 의미를 지정해 함수로 생성해야 함.
- sklearn.metrics에서 make_scorer 패키지 import해서 NMAE는 낮은 값이 더 좋은 평가지수 라는 로직 입력  
`make_scorer(nmae_func, great_is_better=False)`

In [2]:
# from sklearn.metrics import make_scorer

# def NMAE(true, pred):
#    mae = np.mean(np.abs(true - pred))
#    score = mae / np.mean(np.abs(true))
#    return score

# nmae_score = make_scorer(NMAE, greater_is_better=False)

- 진행할 수 있는 모든 모델에 대해 학습 및 예측 수행하여 좋은 결과 가지는 모델 선정함
  - 좋은 모델 선택하는 **개수**의 기준이 무엇이었는가? : 0.9이하의 값을 선정한 듯.
  - 파라미터 튜닝 없이 단순 학습만 진행함: 이게 많은 모델의 값 확인할 수 있던 비결일수도, 단순해서 시간 소요 적음
  - 처음에 8개 선정(ETR + RF + GBR + XGB + LGB + HGB + NGB + CAT) 후 LGB, HGB는 제외함. 왜지?
    - 상관관계, 다양성 고려: 앙상블(Voting)의 핵심은 "서로 다른 예측을 하는 모델들이 모여 오차를 상쇄하는 것"입니다. LGBM과 HistGBR은 모두 히스토그램 기반 알고리즘을 사용하므로, 이미 포함된 CAT(CatBoost)이나 XGB(XGBoost)와 예측 패턴이 너무 비슷했을 수 있습니다. 비슷한 모델이 많아지면 특정 방향으로 편향(Bias)이 생길 수 있어 제외한 것으로 보입니다
    - 과적합 우려: 데이터 샘플 수가 적은 경우, 복잡한 부스팅 모델인 LGBM은 리더보드 점수(Public Score)에서 오히려 성능이 튀거나 과적합될 가능성이 높음
  
- 소프트 보팅 사용 이유는?
  - 회귀에서 소프트 보팅은 각 모델이 예측한 값들의 평균을 내는 방식.
  - 하드 보팅은 분류에서 다수결을 따지는 방식
  - 극단치 중화: 어떤 모델은 과하게 높게 예측하고, 어떤 모델은 낮게 예측할 때 이들의 평균을 구하면 오차가 상쇄되어 훨씬 안정적인 예측이 가능
  - 가중치 활용 가능: 단순히 산술 평균을 낼 수도 있지만, 성능이 더 좋은 모델(예: CAT)에 더 높은 가중치를 주는 '가중 산술 평균' 방식을 사용할 수 있어 하드 보팅보다 훨씬 유연하고 정교함.
